## Rolling Window Portfolio Transformer

Kelly et al. (2025) Section 4.3 rolling window protocol. A 60 month training window advances one month at a time. At each step the model retrains from scratch on the window and produces one out of sample prediction for the next month. Seed averaging across independent initialisations. No validation set or early stopping because the short window provides natural regularisation.

### 1. Setup

In [1]:
import gc
import json
import pickle
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch 2.12.0+cu130, device: cuda
GPU: NVIDIA GeForce RTX 2080 Super with Max-Q Design


### 2. Configuration

In [3]:
# Country
country = 'IND'
data_dir = Path('../data/processed') / country
results_dir = Path('../results') / country / 'rolling'
results_dir.mkdir(parents = True, exist_ok = True)

with open(data_dir / f'{country}_metadata.json') as f: meta = json.load(f)
char_cols = meta['char_cols']
d = meta['d']

firm_lookup = pd.read_parquet(data_dir / f'{country}_firm_lookup.parquet')
id_to_gvkey = dict(zip(firm_lookup['id'], firm_lookup.get('gvkey', firm_lookup['id'])))

# Architecture
n_blocks = 2
n_heads = 1
d_ff = 256

# Rolling window
window_size = 60
n_epochs_per_window = 20
lr = 1e-4
weight_decay = 1e-3
grad_clip = 1.0
n_seeds = 3

# Portfolio
rebalance_freq = 6
tc_bps = 25

variant_list = ['identity', 'linear', 'ple', 'periodic', 'fourier', 'magnitude_dir']

print(f'country: {country}, d = {d}')
print(f'window: {window_size} months, {n_epochs_per_window} epochs/window, {n_seeds} seeds')

country: IND, d = 187
window: 60 months, 20 epochs/window, 3 seeds


### 3. Data Loading

Load all months from train, val, and test parquets into a single chronological sequence. The rolling window selects from this sequence directly.

In [ ]:
def load_all_months():
	"""Load train + val + test into one chronological months dict."""
	all_months = {}
	for split in ['train', 'val', 'test']:
		p = data_dir / f'{country}_{split}.parquet'
		if not p.exists(): continue
		df = pd.read_parquet(p)
		df['eom'] = pd.to_datetime(df['eom'])
		for eom, g in df.groupby('eom'):
			x = g[char_cols].values.astype(np.float32)
			r = g['ret_exc_lead1m'].values.astype(np.float32)
			ids = g['id'].values
			hr = np.isfinite(r)
			if hr.sum() >= 5:
				all_months[eom] = {'x': x[hr], 'r': r[hr], 'ids': ids[hr]}
	return all_months


def month_to_gpu(m):
	return {
		'x': torch.tensor(m['x'], dtype = torch.float32, device = device),
		'r': torch.tensor(m['r'], dtype = torch.float32, device = device),
		'ids': m['ids'],
	}


all_months = load_all_months()
sorted_dates = sorted(all_months.keys())
n_total = len(sorted_dates)
n_oos = n_total - window_size

nf = np.mean([m['x'].shape[0] for m in all_months.values()])
print(f'total months: {n_total}')
print(f'first: {sorted_dates[0].date()}, last: {sorted_dates[-1].date()}')
print(f'OOS months: {n_oos} (from {sorted_dates[window_size].date()})')
print(f'~{nf:.0f} firms/month')

### 4. Encoding Variants

In [ ]:
class IdentityEncoder(nn.Module):
	def forward(self, x): return x

class LinearEncoder(nn.Module):
	def __init__(self, n):
		super().__init__()
		self.w = nn.Parameter(torch.ones(n))
		self.b = nn.Parameter(torch.zeros(n))
	def forward(self, x): return x * self.w + self.b

class PLEEncoder(nn.Module):
	def __init__(self, n, bins = 16):
		super().__init__()
		bd = torch.linspace(-0.5, 0.5, bins + 1)
		self.register_buffer('lo', bd[:-1])
		self.register_buffer('hi', bd[1:])
		self.w = nn.Parameter(torch.zeros(n, bins))
	def forward(self, x):
		a = torch.clamp((x.unsqueeze(-1) - self.lo) / (self.hi - self.lo + 1e-8), 0, 1)
		return x + (a * self.w.unsqueeze(0)).sum(-1)

class PeriodicEncoder(nn.Module):
	def __init__(self, n, nf = 8):
		super().__init__()
		self.om = nn.Parameter(torch.randn(n, nf))
		self.ph = nn.Parameter(torch.randn(n, nf) * 0.1)
		self.c = nn.Parameter(torch.zeros(n, nf))
	def forward(self, x):
		return x + (torch.sin(x.unsqueeze(-1) * self.om.unsqueeze(0) + self.ph.unsqueeze(0)) * self.c.unsqueeze(0)).sum(-1)

class FourierEncoder(nn.Module):
	def __init__(self, n, nf = 8):
		super().__init__()
		self.register_buffer('freq', torch.arange(1, nf + 1, dtype = torch.float32) * torch.pi)
		self.a = nn.Parameter(torch.zeros(n, nf))
		self.b = nn.Parameter(torch.zeros(n, nf))
	def forward(self, x):
		s = x.unsqueeze(-1) * self.freq
		return x + (torch.sin(s) * self.a.unsqueeze(0) + torch.cos(s) * self.b.unsqueeze(0)).sum(-1)

class MagnitudeDirectionEncoder(nn.Module):
	def __init__(self, n):
		super().__init__()
		self.wp = nn.Parameter(torch.ones(n))
		self.wn = nn.Parameter(torch.ones(n))
		self.b = nn.Parameter(torch.zeros(n))
	def forward(self, x): return F.relu(x) * self.wp - F.relu(-x) * self.wn + self.b

def build_encoder(v, n):
	enc = {'identity': IdentityEncoder, 'linear': LinearEncoder, 'ple': PLEEncoder,
		'periodic': PeriodicEncoder, 'fourier': FourierEncoder, 'magnitude_dir': MagnitudeDirectionEncoder}
	return enc[v]() if v == 'identity' else enc[v](n)

### 5. Kelly Architecture

In [ ]:
class AttentionHead(nn.Module):
	def __init__(self, n, s):
		super().__init__()
		self.w = nn.Parameter(torch.randn(n, n) * s)
		self.v = nn.Parameter(torch.randn(n, n) * s)
		self.sc = 1.0 / np.sqrt(n)
	def forward(self, y):
		return F.softmax((y @ self.w @ y.t()) * self.sc, dim = -1) @ (y @ self.v)

class TransformerBlock(nn.Module):
	def __init__(self, n, h, ff, s):
		super().__init__()
		self.heads = nn.ModuleList([AttentionHead(n, s) for _ in range(h)])
		self.w1 = nn.Parameter(torch.randn(n, ff) * (1.0 / ff))
		self.b1 = nn.Parameter(torch.zeros(ff))
		self.w2 = nn.Parameter(torch.randn(ff, n) * s)
		self.b2 = nn.Parameter(torch.zeros(n))
	def forward(self, y):
		y = sum(h(y) for h in self.heads) + y
		return F.relu(y @ self.w1 + self.b1) @ self.w2 + self.b2 + y

class PortfolioTransformer(nn.Module):
	def __init__(self, n, nb, nh, ff, enc):
		super().__init__()
		self.enc = enc
		s = 1.0 / n
		self.blocks = nn.ModuleList([TransformerBlock(n, nh, ff, s) for _ in range(nb)])
		self.lam = nn.Parameter(torch.randn(n) * s)
	def forward(self, x):
		y = self.enc(x)
		for b in self.blocks: y = b(y)
		return y @ self.lam
	def msrr_loss(self, x, r): return (1.0 - self.forward(x) @ r) ** 2

### 6. Rolling Window Training

For each out of sample month, the model trains from scratch on the preceding 60 months, produces one prediction, and discards the model. No validation set or early stopping. The short window is the regularisation.

In [ ]:
def train_one_window(variant, window_gpu, n_epochs, seed):
	"""Train from scratch on one 60 month window."""
	torch.manual_seed(seed)
	np.random.seed(seed)
	model = PortfolioTransformer(d, n_blocks, n_heads, d_ff, build_encoder(variant, d).to(device)).to(device)
	opt = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)
	keys = list(window_gpu.keys())
	nm = len(keys)

	for ep in range(n_epochs):
		model.train()
		for idx in np.random.permutation(nm):
			opt.zero_grad()
			loss = model.msrr_loss(window_gpu[keys[idx]]['x'], window_gpu[keys[idx]]['r'])
			loss.backward()
			nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
			opt.step()
	return model


@torch.no_grad()
def predict_oos(model, oos_month_gpu):
	model.eval()
	return model(oos_month_gpu['x']).cpu().numpy()


def train_variant(variant):
	print(f'\nVariant: {variant}')
	vdir = results_dir / variant
	vdir.mkdir(parents = True, exist_ok = True)
	t0 = time.time()

	# Build GPU cache for all months
	gpu_cache = {eom: month_to_gpu(all_months[eom]) for eom in sorted_dates}

	oos_predictions = {}
	window_losses = []

	for t_idx in range(window_size, n_total):
		oos_date = sorted_dates[t_idx]
		window_dates = sorted_dates[t_idx - window_size : t_idx]
		window_gpu = {d: gpu_cache[d] for d in window_dates}
		oos_gpu = gpu_cache[oos_date]

		# Train n_seeds models, collect predictions
		seed_weights = []
		seed_sdf = []
		for seed in range(n_seeds):
			model = train_one_window(variant, window_gpu, n_epochs_per_window, seed)
			w = predict_oos(model, oos_gpu)
			seed_weights.append(w)
			seed_sdf.append(float(w @ all_months[oos_date]['r']))
			del model

		# Average across seeds (L1 normalised for ranking, raw for SDF)
		w_sum = np.zeros(len(seed_weights[0]))
		nv = 0
		for w in seed_weights:
			w64 = w.astype(np.float64)
			a = np.abs(w64).sum()
			if a > 1e-10:
				w_sum += w64 / a
				nv += 1
		if nv > 0:
			oos_predictions[oos_date] = {
				'w': (w_sum / nv).astype(np.float32),
				'ids': all_months[oos_date]['ids'],
				'r': all_months[oos_date]['r'],
				'sdf_ret': float(np.mean(seed_sdf)),
			}

		if (t_idx - window_size + 1) % 12 == 0 or t_idx == window_size:
			elapsed = time.time() - t0
			pct = (t_idx - window_size + 1) / n_oos * 100
			print(f'  {t_idx - window_size + 1}/{n_oos} OOS months ({pct:.0f}%) '
				f'| {oos_date.date()} | {elapsed / 60:.1f} min')
			sys.stdout.flush()

	del gpu_cache
	gc.collect()
	torch.cuda.empty_cache()

	elapsed = time.time() - t0
	print(f'  done: {len(oos_predictions)} OOS months in {elapsed / 60:.1f} min')

	# Save
	with open(vdir / f'{variant}_{country}_oos_weights.pkl', 'wb') as f:
		pickle.dump(oos_predictions, f)

	return oos_predictions, elapsed

### 7. Evaluation

In [ ]:
def oos_rank_corr(preds):
	corrs = []
	for eom in sorted(preds.keys()):
		m = preds[eom]
		if len(m['w']) < 10: continue
		c, _ = spearmanr(m['w'], m['r'])
		if not np.isnan(c): corrs.append(c)
	return float(np.mean(corrs)) if corrs else 0.0, corrs


def quintile_sim(preds, rf = 6, tc = 25):
	keys = sorted(preds.keys())
	if not keys: return np.array([]), []
	rset = set(keys[::rf])
	ml = []
	li = set()
	si = set()
	pl = set()
	ps = set()
	hl = []
	for eom in keys:
		m = preds[eom]
		w = m['w']
		r = m['r']
		ids = m['ids']
		tcv = 0.0
		if eom in rset:
			nq = max(1, int(len(w) * 0.20))
			so = np.argsort(w)
			li = set(ids[so[::-1][:nq]].tolist())
			si = set(ids[so[:nq]].tolist())
			to = (len(li - pl) + len(pl - li) + len(si - ps) + len(ps - si)) / max(nq, 1)
			tcv = to * tc / 10000.0
			pl = li
			ps = si
			hl.append({'eom': str(eom),
				'long': [{'id': i, 'gvkey': id_to_gvkey.get(i, '')} for i in sorted(li)],
				'short': [{'id': i, 'gvkey': id_to_gvkey.get(i, '')} for i in sorted(si)]})
		if not li: continue
		il = ids.tolist()
		lr = r[np.array([i in li for i in il])]
		sr = r[np.array([i in si for i in il])]
		ml.append((float(lr.mean()) if len(lr) > 0 else 0) - (float(sr.mean()) if len(sr) > 0 else 0) - tcv)
	return np.array(ml), hl


def score_weighted_sim(preds):
	ml = []
	for eom in sorted(preds.keys()):
		w = preds[eom]['w'].astype(np.float64)
		r = preds[eom]['r'].astype(np.float64)
		ww = w - w.mean()
		a = np.abs(ww).sum()
		if a > 1e-10: ml.append(float((ww / a) @ r))
	return np.array(ml)


def sdf_sim(preds):
	return np.array([preds[k]['sdf_ret'] for k in sorted(preds.keys()) if 'sdf_ret' in preds[k]])


def portfolio_metrics(rets, ppy = 12):
	if len(rets) == 0: return {}
	tw = float((1 + rets).prod())
	ann_ret = -1.0 if tw <= 0 else float(tw ** (ppy / len(rets)) - 1)
	av = float(rets.std() * np.sqrt(ppy))
	sr = ann_ret / max(av, 1e-8)
	se = float(np.sqrt((1 + 0.5 * sr ** 2) / len(rets)))
	pk = np.maximum.accumulate(np.cumprod(1 + rets))
	dd = float(((pk - np.cumprod(1 + rets)) / pk).max()) if len(pk) > 0 else 0
	return {'ann_ret': ann_ret, 'ann_vol': av, 'sharpe': sr, 'se_sharpe': se, 'max_dd': dd, 'n_months': len(rets)}


def evaluate_variant(preds, vname, vdir):
	print(f'\n  {vname} ({len(preds)} OOS months)')

	rc, rc_monthly = oos_rank_corr(preds)
	print(f'  Rank corr: {rc:.4f}')

	qr, holdings = quintile_sim(preds)
	qm = portfolio_metrics(qr)
	print(f'  Quintile   sharpe: {qm.get("sharpe", 0):.4f} (se {qm.get("se_sharpe", 0):.4f})  ret: {qm.get("ann_ret", 0) * 100:.2f}%')

	swr = score_weighted_sim(preds)
	swm = portfolio_metrics(swr)
	print(f'  Score wt   sharpe: {swm.get("sharpe", 0):.4f} (se {swm.get("se_sharpe", 0):.4f})  ret: {swm.get("ann_ret", 0) * 100:.2f}%')

	sdfr = sdf_sim(preds)
	sdfm = portfolio_metrics(sdfr)
	print(f'  SDF        sharpe: {sdfm.get("sharpe", 0):.4f} (se {sdfm.get("se_sharpe", 0):.4f})  ret: {sdfm.get("ann_ret", 0) * 100:.2f}%')

	np.save(vdir / f'{vname}_{country}_quintile.npy', qr)
	np.save(vdir / f'{vname}_{country}_scorewt.npy', swr)
	np.save(vdir / f'{vname}_{country}_sdf.npy', sdfr)
	with open(vdir / f'{vname}_{country}_holdings.json', 'w') as f:
		json.dump(holdings, f, indent = 2, default = str)
	with open(vdir / f'{vname}_{country}_rank_corrs.json', 'w') as f:
		json.dump({'mean': rc, 'monthly': rc_monthly}, f, default = float)

	return {'variant': vname, 'rank_corr': rc,
		'quintile': qm, 'score_weighted': swm, 'sdf': sdfm}

### 8. Per Variant Training

In [5]:
results = {}

In [ ]:
preds_identity, time_identity = train_variant('identity')
results['identity'] = evaluate_variant(preds_identity, 'identity', results_dir / 'identity')
results['identity']['time_min'] = time_identity / 60

In [ ]:
preds_linear, time_linear = train_variant('linear')
results['linear'] = evaluate_variant(preds_linear, 'linear', results_dir / 'linear')
results['linear']['time_min'] = time_linear / 60

In [ ]:
preds_ple, time_ple = train_variant('ple')
results['ple'] = evaluate_variant(preds_ple, 'ple', results_dir / 'ple')
results['ple']['time_min'] = time_ple / 60

In [ ]:
preds_periodic, time_periodic = train_variant('periodic')
results['periodic'] = evaluate_variant(preds_periodic, 'periodic', results_dir / 'periodic')
results['periodic']['time_min'] = time_periodic / 60

In [ ]:
preds_fourier, time_fourier = train_variant('fourier')
results['fourier'] = evaluate_variant(preds_fourier, 'fourier', results_dir / 'fourier')
results['fourier']['time_min'] = time_fourier / 60

In [ ]:
preds_magnitude_dir, time_magnitude_dir = train_variant('magnitude_dir')
results['magnitude_dir'] = evaluate_variant(preds_magnitude_dir, 'magnitude_dir', results_dir / 'magnitude_dir')
results['magnitude_dir']['time_min'] = time_magnitude_dir / 60

### 9. Results

In [6]:
print(f'Rolling Window Encoding Comparison: {country}')
print(f'window = {window_size} months, {n_epochs_per_window} epochs, {n_seeds} seeds')
print()
print(f'{"Variant":<18} {"Corr":>6} {"Q":>7} {"SW":>7} {"SDF":>7} {"Ret":>7} {"Time":>5}')
print()
for v, r in sorted(results.items(), key = lambda x: -x[1]['rank_corr']):
	q = r.get('quintile', {})
	sw = r.get('score_weighted', {})
	sd = r.get('sdf', {})
	print(f'{v:<18} {r["rank_corr"]:6.4f} {q.get("sharpe", 0):7.3f} '
		f'{sw.get("sharpe", 0):7.3f} {sd.get("sharpe", 0):7.3f} '
		f'{q.get("ann_ret", 0) * 100:6.2f}% {r.get("time_min", 0):4.0f}m')

with open(results_dir / f'{country}_rolling_comparison.json', 'w') as f:
	json.dump(results, f, indent = 2, default = lambda x: float(x) if hasattr(x, '__float__') else str(x))
print(f'\nSaved: {results_dir / f"{country}_rolling_comparison.json"}')

Rolling Window Encoding Comparison: IND
window = 60 months, 20 epochs, 3 seeds

Variant              Corr       Q      SW     SDF     Ret  Time


Saved: ..\results\IND\rolling\IND_rolling_comparison.json


In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (14, 10))
fig.suptitle(f'{country}: Rolling Window ({window_size}m) Encoding Comparison', fontsize = 14)
vs = list(results.keys())
lb = [v.replace('_', ' ').title() for v in vs]
x = np.arange(len(vs))

# Rank correlation
axes[0,0].bar(x, [results[v]['rank_corr'] for v in vs])
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(lb, rotation = 25, ha = 'right')
axes[0,0].set_title('OOS Rank Correlation')
axes[0,0].grid(axis = 'y', alpha = 0.3)

for i, (key, title) in enumerate([('quintile', 'Quintile Sharpe'), ('score_weighted', 'Score Weighted'), ('sdf', 'SDF Sharpe')]):
	ax = axes[(i + 1) // 2, (i + 1) % 2]
	sh = [results[v].get(key, {}).get('sharpe', 0) for v in vs]
	se = [results[v].get(key, {}).get('se_sharpe', 0) for v in vs]
	ax.bar(x, sh, yerr = se, capsize = 3)
	ax.set_xticks(x)
	ax.set_xticklabels(lb, rotation = 25, ha = 'right')
	ax.set_title(title)
	ax.grid(axis = 'y', alpha = 0.3)

plt.tight_layout()
plt.savefig(results_dir / f'{country}_rolling_comparison.png', dpi = 150, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (10, 5))
for v in vs:
	p = results_dir / v / f'{v}_{country}_quintile.npy'
	if p.exists():
		ax.plot(np.cumprod(1 + np.load(p)), label = v.replace('_', ' ').title())
ax.set_title(f'{country}: Cumulative Wealth (Rolling Window, Quintile)')
ax.set_xlabel('Month')
ax.set_ylabel('Wealth')
ax.legend()
ax.grid(alpha = 0.3)
plt.tight_layout()
plt.savefig(results_dir / f'{country}_rolling_cumulative.png', dpi = 150, bbox_inches = 'tight')
plt.show()